# Práctica 1: Redes Bayesianas para predecir abandono de clientes

**Estudiante:** Adrian Torrico  \n**Materia:** Minería de Datos

En este notebook desarrollo y evalúo dos Redes Bayesianas discretas para estimar la probabilidad de abandono (churn) de clientes de Internet.

## 0. Preparación

Subo `customers (2).csv` al entorno de Colab antes de ejecutar las celdas. La versión fijada de `pgmpy` mantiene la compatibilidad del código y hace el resultado reproducible.

In [ ]:
!pip install -q pgmpy==0.1.25

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import BayesianEstimator, HillClimbSearch, BicScore
from pgmpy.inference import VariableElimination
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')

## Actividad 1. Exploración de los datos

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

DATA_FILE = 'customers (2).csv'
DATA_URL = 'https://raw.githubusercontent.com/adriantorrico126/practica-1-redes-bayesianas/main/customers%20%282%29.csv'
if not Path(DATA_FILE).exists():
    urlretrieve(DATA_URL, DATA_FILE)
    print('CSV descargado desde el repositorio.')

df = pd.read_csv(DATA_FILE)
print('Dimensiones:', df.shape)
display(df.head())
display(df.info())
display(df.isna().sum().to_frame('Valores faltantes'))
display(df.describe().T)

fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.countplot(data=df, x='Churn', hue='Churn', legend=False, ax=axes[0], palette={'No':'#3B82F6','Yes':'#EF4444'})
axes[0].set_title('Distribución de abandono')
sns.countplot(data=df, x='Contract_Type', hue='Churn', ax=axes[1])
axes[1].set_title('Abandono según tipo de contrato')
plt.tight_layout()
plt.show()

Interpretación: reviso el balance de la variable objetivo, los faltantes y el comportamiento de las variables. `Internet_Service` tiene valores faltantes; los conservo como una categoría explícita (`Unknown`) para no eliminar clientes ni inventar información.

## Actividad 2. Preparación y discretización

In [ ]:
def discretize(df):
    d = df.copy()
    d['Internet_Service'] = d['Internet_Service'].fillna('Unknown')
    d['Age_Group'] = pd.cut(d['Age'], [17,30,45,60,70], labels=['Young','Adult','Mature','Senior'], include_lowest=True).astype(str)
    d['Tenure_Group'] = pd.cut(d['Tenure_Months'], [-1,12,24,48,np.inf], labels=['Low','Medium','Established','Loyal'], include_lowest=True).astype(str)
    d['Amount_Group'] = pd.qcut(d['Monthly_Amount'], 3, labels=['Low','Medium','High'], duplicates='drop').astype(str)
    d['Complaints_Group'] = pd.cut(d['Complaints'], [-1,0,2,np.inf], labels=['None','Few','Many'], include_lowest=True).astype(str)
    d['Satisfaction_Group'] = pd.cut(d['Satisfaction'], [0,4,7,10], labels=['Low','Medium','High'], include_lowest=True).astype(str)
    cols = ['Age_Group','Tenure_Group','Amount_Group','Contract_Type','Payment_Method','Technical_Support','Internet_Service','Complaints_Group','Satisfaction_Group','Churn']
    return d[cols].astype(str)

data = discretize(df)
display(pd.DataFrame({'Variable':data.columns, 'Categorías':[', '.join(sorted(data[c].unique())) for c in data.columns]}))

fig, axes = plt.subplots(1, 2, figsize=(12,4))
for ax, col, order in [(axes[0], 'Satisfaction_Group', ['Low','Medium','High']), (axes[1], 'Tenure_Group', ['Low','Medium','Established','Loyal'])]:
    rate = pd.crosstab(data[col], data['Churn'], normalize='index')['Yes'].mul(100).reindex(order)
    rate.plot(kind='bar', ax=ax, color='#EF4444')
    ax.set_title(f'Tasa de abandono por {col}')
    ax.set_ylabel('Abandono (%)'); ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

## División de entrenamiento y prueba

In [ ]:
train, test = train_test_split(data, test_size=0.25, stratify=data['Churn'], random_state=RANDOM_STATE)
print(f'Entrenamiento: {len(train)} clientes | Prueba: {len(test)} clientes')

## Actividad 3. Red Bayesiana propuesta

Propongo que contrato, soporte, satisfacción, antigüedad, reclamos y servicio de Internet influyan directamente en el abandono. También conecto edad, monto y reclamos con satisfacción, y contrato con antigüedad. Es una estructura explicable basada en el proceso de negocio.

In [ ]:
manual_edges = [('Age_Group','Satisfaction_Group'), ('Amount_Group','Satisfaction_Group'), ('Complaints_Group','Satisfaction_Group'), ('Contract_Type','Tenure_Group'), ('Contract_Type','Churn'), ('Technical_Support','Churn'), ('Satisfaction_Group','Churn'), ('Tenure_Group','Churn'), ('Complaints_Group','Churn'), ('Internet_Service','Churn')]
manual = BayesianNetwork(manual_edges)
manual.add_nodes_from(data.columns)
manual.fit(train, estimator=BayesianEstimator, prior_type='BDeu', equivalent_sample_size=10)

def draw_network(model, title):
    g = nx.DiGraph(model.edges())
    plt.figure(figsize=(10,6))
    pos = nx.spring_layout(g, seed=RANDOM_STATE, k=1.3)
    nx.draw(g, pos, with_labels=True, node_color=['#FCA5A5' if n=='Churn' else '#BFDBFE' for n in g.nodes()], node_size=2200, arrowsize=20, font_size=8, font_weight='bold')
    plt.title(title); plt.show()

draw_network(manual, 'Red Bayesiana propuesta')
for cpd in manual.get_cpds():
    print(cpd)
    print('-'*70)

Las salidas anteriores son las tablas de probabilidad condicional (CPD) estimadas con suavizado BDeu. El suavizado evita probabilidades exactamente cero para combinaciones poco frecuentes.

## Actividad 4. Estructura automática con Hill Climbing

In [ ]:
hc = HillClimbSearch(train)
learned_structure = hc.estimate(scoring_method=BicScore(train), max_indegree=3, max_iter=1000, show_progress=False)
automatic = BayesianNetwork(learned_structure.edges())
automatic.add_nodes_from(data.columns)
automatic.fit(train, estimator=BayesianEstimator, prior_type='BDeu', equivalent_sample_size=10)

print('Aristas encontradas:')
print(list(automatic.edges()))
draw_network(automatic, 'Red Bayesiana aprendida con Hill Climbing')
for cpd in automatic.get_cpds():
    print(cpd)
    print('-'*70)

## Actividad 5. Inferencia probabilística

In [ ]:
inference = VariableElimination(automatic)
evidence = {'Contract_Type':'Monthly', 'Technical_Support':'No', 'Satisfaction_Group':'Low', 'Tenure_Group':'Low'}
query = inference.query(variables=['Churn'], evidence=evidence, show_progress=False)
print('Evidencia:', evidence)
print(query)
print(f"Probabilidad de abandono: {query.values[list(query.state_names['Churn']).index('Yes')]:.2%}")

## Actividad 6. Evaluación de los modelos

In [ ]:
def evaluate(model, test):
    features = [c for c in test.columns if c != 'Churn']
    pred = model.predict(test[features])['Churn']
    y = test['Churn']
    return {'Accuracy':accuracy_score(y,pred), 'Precision':precision_score(y,pred,pos_label='Yes',zero_division=0), 'Recall':recall_score(y,pred,pos_label='Yes',zero_division=0), 'F1-score':f1_score(y,pred,pos_label='Yes',zero_division=0)}, confusion_matrix(y,pred,labels=['No','Yes']), classification_report(y,pred,zero_division=0)

manual_metrics, manual_cm, manual_report = evaluate(manual, test)
auto_metrics, auto_cm, auto_report = evaluate(automatic, test)
metrics = pd.DataFrame([manual_metrics, auto_metrics], index=['Red propuesta','Hill Climbing']).round(4)
display(metrics)
print('Matriz de confusión — Red propuesta [No, Yes]:\n', manual_cm)
print('Matriz de confusión — Hill Climbing [No, Yes]:\n', auto_cm)
print(auto_report)
metrics.plot(kind='bar', figsize=(10,4), ylim=(0,1), rot=0)
plt.title('Comparación de métricas'); plt.ylabel('Valor'); plt.tight_layout(); plt.show()

## Actividad 7. Comparación y conclusión

La red propuesta usa conocimiento del problema y por ello sus relaciones son fáciles de justificar y comunicar. La red de Hill Climbing explora los datos y puede descubrir asociaciones no anticipadas, pero una arista no debe interpretarse automáticamente como causalidad. Comparo las métricas en el conjunto de prueba y priorizo Recall/F1 para churn, porque Accuracy puede verse favorecida por la clase mayoritaria de clientes que no abandonan. Una mejora posterior sería ajustar el umbral de decisión, incorporar más datos y validar mediante particiones repetidas.